# Transparency Portal

Brazilian municipal, state and federal governments must, by law, provide data on public spending, budget, and use of federal funds. Of one the tools used for that is "Portal da Transparência" (Transparency Portal), a site that provides such data for public access.

While every municipality, state and federal governments have their own individual Portal da Transparencia, the system behind it varies. Publicenter is a brazilian company that develops a "Portal da Transparencia" system that is used my many brazilian municipalities, including my hometown.

Due to this connection, this project aims to extract information from Publicent's Portal da Transparencia and ease its visualization.

## The study case

The chosen municipality is Lagoa Formosa, a small town located in the interior of the state of Minas Gerais and my hometown.

Lagoa Formosa's Portal da Transparencia can be accessed by this link: [Lagoa Formosa's Portal da Transparência](https://transparencia.lagoaformosa.mg.gov.br/#/transparencia)

This site offers data on many different aspects of public administration, such as, for instance, positions and wages, public contracts, statistical reports, etc., however this project will focus only on public expense data, which is divided budgetary and off-budgetary expenditure.

## The first step: Undestanding HTTP requests

We will begin this project by undestanding how the front-end interact with the system's back-end.
For this we will be using our browser dev tools to inspect HTTP requests and their responses. Lets start with the budgetary expenses:

<p align="center">
  <a href="documentation\images\00-first-header.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
  <a href="documentation\images\01-first-response.png">
    <img src="documentation\images\01-first-response.png" width="45%">
  </a>
</p>

From this we can get the request url:
`https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina=20&pagina=1&termoBase64=&indAgrupamento=FOR&datInicio=2026-07-01&datFim=2026-07-31&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho=`

By analysing it we can notice some interesting query parameters, more specifically:
- `elementosPorPagina=` which decribes how many elements will be returned
- `pagina=`, which describes the page requested
- `datInicio=`, that is the starting date
- `datFim=`, that is the ending date

By analysing the response we found tree important keys:
- `"content"`, which is a list of entries
- `"totalElements"`, that is a number of the total number of data entries
- `"totalPages"`, that lists the number of pages
- `"size"`, that is the size of each page

With that we can begin playing with HTTP requests:


In [1]:
# we opted to use the httpx library instead of the requests library
import httpx

elementsPerPage = 20
page = 1
initialDate = '2010-01-01'
finalDate = '2026-07-29'

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40487, totalPages: 2025, size: 20


## The second step: Extracting data from Portal da Transparência

From this we can see that we have a total of 40487 entries distributed among 2025 pages.

We can play with the query parameters to try to reduce the amount of pages, and consequently reduce the number of requests we'll have to make to obtain all data.

Of course, we need to be careful to not trigger a timeout or any other block.

In [2]:
elementsPerPage = 10000

url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='

response = httpx.get(url, verify=False)
r = response.json()
totalElements = r['totalElements']
totalPages = r['totalPages']
size = r['size']
print(f'totalElements: {totalElements}, totalPages: {totalPages}, size: {size}')

totalElements: 40487, totalPages: 5, size: 10000


We can see that by setting the number of elements per page as 10000 we reduced the number of pages from 2025 to only 5.

There's not a magical number, so we always need to verify how many elements per page the systems support.

Anyway, we can now capture the data:

In [3]:
elementsPerPage = 10000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):

    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaDetalhada?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&indAgrupamento=FOR&datInicio={initialDate}&datFim={finalDate}&desNatureza=&codAdministracao=1&filtrarApenasCovid=false&numEmpenho='
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']
    
    page += 1

    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')




totalElements: 40487, elements obtained: 10000
totalElements: 40487, elements obtained: 20000
totalElements: 40487, elements obtained: 30000
totalElements: 40487, elements obtained: 40000
totalElements: 40487, elements obtained: 40487


## Third step: converting data for a tabular format

Now we have a list made of 40487 dictionaries, however it is still hard to work with data in this structure. For this reason we will convert it for a tabular format.

We could assume that every dictionary has the same keys to ease the extraction, but we'll check every entry to garantee no data is left behind.

To garantee we'll not need to request the data again, we'll also save it as an csv file, so we can open it latter if needed.

In [4]:
dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedBudgetaryData.append(row)

import csv
with open ('extractedBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedBudgetaryData)

## Doing the same for the off-budgetary expenditure.

We can notice that the request url is pretty similar: `https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina=20&pagina=1&termoBase64=&datInicio=2026-07-06&datFim=2026-07-31`

So we only need to work with the some parameters as before.

In [5]:
elementsPerPage = 1000
initialDate = '2010-01-01'
finalDate = '2026-07-29'

content = []
page, totalPages, totalElements = 1, 1, 1

while (len(content) < totalElements) and (page <= totalPages):
    
    url = f'https://transparencia.lagoaformosa.mg.gov.br/publico/despesaExtra?elementosPorPagina={elementsPerPage}&pagina={page}&termoBase64=&datInicio={initialDate}&datFim={finalDate}'
    response = httpx.get(url, verify=False)
    r = response.json()
    content.extend(r['content'])
    
    if page == 1:
        totalElements = r['totalElements']
        totalPages = r['totalPages']    
    page += 1
    print(f'totalElements: {totalElements}, elements obtained: {len(content)}')

dictKeys = set()

for i in range(len(content)):
    dictKeys.update(content[i].keys())

extractedOffBudgetaryData = [list(dictKeys)]
for i in range(len(content)):
    row = []
    for j in dictKeys:
        row.append(content[i][j])
    extractedOffBudgetaryData.append(row)

import csv
with open ('extractedOffBudgetaryData.csv', 'w', newline="", encoding='utf-8') as file:
    writer = csv.writer(file, delimiter=';')
    writer.writerows(extractedOffBudgetaryData)


totalElements: 2814, elements obtained: 1000
totalElements: 2814, elements obtained: 2000
totalElements: 2814, elements obtained: 2814


## Fourth step: Processing data
Now that we have extracted the data, we can work with it, and for this we have many options. 

In this project we'll use two: pandas and metabase:
- Pandas is a powerfull library focused on the manipulation, cleaning, and analysis of structured tabular data.
- Jupyter SQL magic allows us to execute SQL queries directly within Jupyter Notebook cells and display results interactively.
- Metabase is a user-friendly Business Intelligence (BI) tool that allows you to explore data, create charts, and build interactive dashboards, with or without SQL code.

In Pandas we'll be manipulating the data directly in this notebook, as for metabase and SQL magic, we'll convert the data into a sqlite database file and import it directly on metabase and process it using SQL Magic.

In [17]:
import pandas as pd
import sqlite3
import hashlib
import re

budgetDf = pd.DataFrame(extractedBudgetaryData[1:], columns=extractedBudgetaryData[0])
offBudgetDf = pd.DataFrame(extractedOffBudgetaryData[1:], columns=extractedOffBudgetaryData[0])


### 4.1 - Understanding and cleaning the data
Now we need to undestand and clean our data. This step is an essential for every data analysis.

Will also make a small data treatment using pandas prior to exporting it to a sqlite file.

Observation: *I noticed that the data contains sensitive data of natural persons, more specifically the CPF, equivalent to the USA's social security number. For this reason we'll replace such data with a sha-256 hash, this way we'll still be able to agregate data without compromising sensitive personal data.*

In [18]:
# Removing Personal data
cpfRegex = re.compile(r'^\d{3}\.\d{3}\.\d{3}-\d{2}$')

def applyHashCpf(df, column):
    mask = df[column].str.match(cpfRegex, na=False)
    df.loc[mask, column] = (
        df.loc[mask, column]
          .apply(lambda x: hashlib.sha256(x.encode("utf-8")).hexdigest())
    )

applyHashCpf(budgetDf, "desDocumentoFornecedor")
applyHashCpf(offBudgetDf, "numDocumentoCredor")

# Removing empty collumns and rows
def dropNaDf(df):
    df.dropna(axis=1, how="all", inplace = True)
    df.dropna(axis=0, how="all", inplace = True)

dropNaDf(budgetDf)
dropNaDf(offBudgetDf)

# Removing collumns where all entries have the same value (as they are irrelevant)
budgetDf = budgetDf.loc[:, budgetDf.nunique() > 1]
offBudgetDf = offBudgetDf.loc[:, offBudgetDf.nunique() > 1]


Checking the data structure

In [21]:
budgetDf

,desLicitacaoEmpenho,desDocumentoFornecedor,vlrPagoFormatado,desModalidade,empenhoFormatado,vlrLiquidoPago,numAnoEmpenho,vlrEmpenhado,mostrarLicitacao,desDocumentoFornecedorFormatado,...,vlrPago,desProcessoEmpenho,vlrDescontoInss,vlrMovimento,numEmpenho,desAgrupamento,desLicitacaoEmpenhoFormatado,totalDesconto,desMovimento,vlrDescontoIrrf
0,000000/0000/01,64886d8459051506c1a9b57935f81f584b5cb00872be59...,"0,00",Dispensada,004143/2026,NaN,2026,400.00,True,***.757.036-**,...,0.00,000000/0000,0.0,"400,00",004143,***.757.036-** - MARCONI ALVES DE OLIVEIRA,000000/0000,0.0,Empenho,0.0
1,000000/0000/01,30.689.142/0001-79,"0,00",Dispensada,004142/2026,NaN,2026,731.00,True,30.689.142/0001-79,...,0.00,000000/0000,0.0,"731,00",004142,30.689.142/0001-79 - MARTINS FERRAMENTAS E PEC...,000000/0000,0.0,Empenho,0.0
2,000000/0000/01,30.689.142/0001-79,"0,00",Dispensada,004141/2026,NaN,2026,731.00,True,30.689.142/0001-79,...,0.00,000000/0000,0.0,"731,00",004141,30.689.142/0001-79 - MARTINS FERRAMENTAS E PEC...,000000/0000,0.0,Empenho,0.0
3,000000/0000/01,19.469.281/0001-54,"0,00",Dispensada,004140/2026,NaN,2026,4250.00,True,19.469.281/0001-54,...,0.00,000000/0000,0.0,"4.250,00",004140,19.469.281/0001-54 - RIBER POCOS ARTESIANOS LTDA,000000/0000,0.0,Empenho,0.0
4,000000/0000/01,30.689.142/0001-79,"0,00",Dispensada,004139/2026,NaN,2026,911.00,True,30.689.142/0001-79,...,0.00,000000/0000,0.0,"911,00",004139,30.689.142/0001-79 - MARTINS FERRAMENTAS E PEC...,000000/0000,0.0,Empenho,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40482,000000/0000/01,6ea64432da291cf0ae52cdb77877fd01d5f61a0a72f49f...,"189,00",Dispensada,000005/2021,189.00,2021,189.00,True,***.320.576-**,...,189.00,000000/0000,0.0,"189,00",000005,***.320.576-** - MARCELO DOMINGUES VIEIRA,000000/0000,0.0,Empenho,0.0
40483,000000/0000/01,ec13026709443837cb4792534b4a5c2bb20f37f75e077a...,"94,50",Dispensada,000004/2021,94.50,2021,94.50,True,***.271.246-**,...,94.50,000000/0000,0.0,"94,50",000004,***.271.246-** - LAURO RIBEIRO DE MENDONÇA,000000/0000,0.0,Empenho,0.0
40484,000000/0000/01,dec0550518da9e657d56e91af3df20d50b7dfb87d2f3dc...,"68,72",Dispensada,000003/2021,68.72,2021,68.72,True,***.619.266-**,...,68.72,000000/0000,0.0,"68,72",000003,***.619.266-** - SEBASTIAO ANSELMO GOMES DOS S...,000000/0000,0.0,Empenho,0.0
40485,000000/0000/01,3aa2e2f1ae793959959e5e4762eabc56ae44d64770396c...,"94,50",Dispensada,000002/2021,94.50,2021,94.50,True,***.872.446-**,...,94.50,000000/0000,0.0,"94,50",000002,***.872.446-** - ADAN CARLOS REZENDE RIBEIRO,000000/0000,0.0,Empenho,0.0


In [22]:
offBudgetDf

,vlrDespesa,desConta,numDocumentCredorFormatado,datMovimento,numDocumentoCredor,codDespesaExtra,desClassificacao,nomCredor,fornecedorFormatado
0,94.12,ISSQN PRESTADORES DE SERVIÇOS,18.602.078/0001-41,2026-07-16,18.602.078/0001-41,2792,218850108,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA
1,60.00,DESCONTO DE ADIANTAMENTO DE VIAGEM DOS SERVIDORES,18.602.078/0001-41,2026-07-10,18.602.078/0001-41,199,218810105,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA
2,198.92,IRRF PRESTADORES DE SERVIÇOS,18.602.078/0001-41,2026-06-30,18.602.078/0001-41,2274,218830104,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA
3,189509.26,PREVIDENCIA MUNICIPAL - SIBEL,23.096.837/0001-81,2026-06-30,23.096.837/0001-81,551,218820101,SISTEMA DE BENEFICIENCIA DOS SERV PUBL MUN LAG...,23.096.837/0001-81 - SISTEMA DE BENEFICIENCIA ...
4,2176.05,IRRF PRESTADORES DE SERVIÇOS,18.602.078/0001-41,2026-06-29,18.602.078/0001-41,2276,218830104,MUNICIPIO DE LAGOA FORMOSA,18.602.078/0001-41 - MUNICIPIO DE LAGOA FORMOSA
...,...,...,...,...,...,...,...,...,...
2809,240.35,PENSAO JUDICIAL,***.244.586-**,2021-01-14,1e4b979551b336e9a117f072baeb0f34889ab49fa4b524...,967,218810110,PAOLA TERENCIO BRAGA,***.244.586-** - PAOLA TERENCIO BRAGA
2810,3885.38,CONTRIBUICAO FILIACAO SINDICAL,23.090.558/0001-00,2021-01-14,23.090.558/0001-00,738,218810113,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS D...,23.090.558/0001-00 - SINDICATO DOS SERVIDORES ...
2811,56690.07,CONVENIO SINDICATO SERVIDORES,23.090.558/0001-00,2021-01-14,23.090.558/0001-00,1206,218810113,SINDICATO DOS SERVIDORES PUBLICOS MUNICIPAIS D...,23.090.558/0001-00 - SINDICATO DOS SERVIDORES ...
2812,645.43,PENSAO JUDICIAL,***.127.666-**,2021-01-14,c4622f2df8dbd8b58513034d510e91845f3f28d782e46a...,1517,218810110,VERA LUCIA OLIVEIRA ABREU SILVA,***.127.666-** - VERA LUCIA OLIVEIRA ABREU SILVA


After analysing the data visually we found the following column have correspondent data: 
- Budgetary Data
    - desDocumentoFornecedorFormatado ~ fornecedorFormatado
    - numEmpenho & numAnoEmpenho = empenhoFormatado
    - desLicitacaoEmpenhoFormatado ~ desLicitacaoEmpenho
    - vlrEmpenhado = vlrEmpenhadoFormatado & vlrMovimento
    - vlrLiquidado = vlrLiquidadoFormatado
    - vlrPago ~ vlrPagoFormatado
- Off Budgetary Data
    - nomCredor & numDocumentCredorFormatado = fornecedorFormatado

Thus they will be removed to avoid redundant data (the ones in the left will be kept)

In [23]:
budgetDf.drop(columns=['fornecedorFormatado', 'empenhoFormatado', 'desLicitacaoEmpenho', 'vlrEmpenhadoFormatado', 'vlrMovimento', 'vlrLiquidadoFormatado', 'vlrPagoFormatado'], inplace = True)
offBudgetDf.drop(columns=['fornecedorFormatado'], inplace = True)

We also found that the column "desModalidade" of budgetDf contains similar data, but i needs to be processed/adjusted:

In [26]:
budgetDf['desModalidade'].unique()

<ArrowStringArray>
[                              'Dispensada',
                      'PregÃ£o EletrÃ´nico',
                        'Tomada de PreÃ§os',
                      'PREGÃƒO ELETRÃ”NICO',
                             'Concorrencia',
                                  'PregÃ£o',
 'INEXIGIBILIDADE CREDENCIA/CHAM. PÃšBLICO',
                          'INEXIGIBILIDADE',
                            'ConcorrÃªncia',
                                 'DISPENSA',
       'INEXIGIBILIDADE POR CREDENCIAMENTO',
                          'Inexigibilidade',
                                 'Dispensa',
                                  'Convite',
                            'Carta Convite']
Length: 15, dtype: str

In [27]:
equivalenceDict = {
    'DISPENSA': 'Dispensa',
    'Dispensada': 'Dispensa',
    'Tomada de PreÃ§os' : 'Tomada de Precos',
    'PREGÃƒO ELETRÃ”NICO' : 'Pregao Eletronico',
    'PregÃ£o EletrÃ´nico' : 'Pregao Eletronico',
    'PregÃ£o' : 'Pregao',
    'INEXIGIBILIDADE' : 'Inexigibilidade',
    'INEXIGIBILIDADE POR CREDENCIAMENTO' : 'Inexigibilidade',
    'INEXIGIBILIDADE CREDENCIA/CHAM. PÃšBLICO' : 'Inexigibilidade',
    'ConcorrÃªncia' : 'Concorrencia', 
    'Carta Convite' : 'Convite'
}

budgetDf["desModalidade"] = budgetDf["desModalidade"].replace(equivalenceDict)
budgetDf['desModalidade'].unique()


<ArrowStringArray>
[         'Dispensa', 'Pregao Eletronico',  'Tomada de Precos',
      'Concorrencia',            'Pregao',   'Inexigibilidade',
           'Convite']
Length: 7, dtype: str

Now the data is good enough to be exported / saved:

In [31]:
# Exporting data

# We'll be rewriting the previously generated CSVs in order to not compromise personal data
budgetDf.to_csv("extractedBudgetaryData.csv", sep=";", index=False)
offBudgetDf.to_csv("extractedOffBudgetaryData.csv", sep=";", index=False)

# We'll be using pandas itself to connect to the sqlite and crete the tables:
dbconn = sqlite3.connect('portalDaTransparencia.sqlite')

budgetDf.to_sql(name='budgetData', con=dbconn, if_exists='replace', index=False)
offBudgetDf.to_sql(name='offBudgetData', con=dbconn, if_exists='replace', index=False)

dbconn.close()

print(f'length budgetDf: {len(budgetDf)}, length budgetDf: {len(offBudgetDf)}')

length budgetDf: 40487, length budgetDf: 2814


If we need to read the data again from the files we can just:

In [ ]:
budgetDf = pd.read_csv("extractedBudgetaryData.csv", sep=";")
offBudgetDf = pd.read_csv("extractedOffBudgetaryData.csv", sep=";")
dbconn = sqlite3.connect('portalDaTransparencia.sqlite')